# Feature Engineering

In [8]:
import pandas as pd
import numpy as np
import joblib
from pathlib import Path
from sentence_transformers import SentenceTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from scipy.sparse import hstack, csr_matrix

#Configuration 
PROCESSED_DATA_DIR = Path("data/02_processed")
FEATURES_DATA_DIR = Path("data/03_features")
CLEAN_DATA_FILE = PROCESSED_DATA_DIR / "clean_cve_data.csv"
SBERT_MODEL_NAME = 'all-MiniLM-L6-v2'

FEATURES_DATA_DIR.mkdir(parents=True, exist_ok=True)
print("Setup complete. Paths and libraries are ready.")

Setup complete. Paths and libraries are ready.


In [9]:
print("Loading clean data...")
df = pd.read_csv(CLEAN_DATA_FILE)
df.head()

Loading clean data...


,CVE_ID,Description,Severity,CVSS_Score,CWE,Year
0,CVE-1999-0006,Buffer overflow in POP servers based on BSD/Qu...,CRITICAL,9.8,NVD-CWE-Other,1999
1,CVE-1999-0011,Denial of Service vulnerabilities in BIND 4.9 ...,MEDIUM,5.4,NVD-CWE-Other,1999
2,CVE-1999-0012,Some web servers under Microsoft Windows allow...,HIGH,7.0,NVD-CWE-Other,1999
3,CVE-1999-0013,Stolen credentials from SSH clients via ssh-ag...,HIGH,8.4,NVD-CWE-Other,1999
4,CVE-1999-0022,Local user gains root privileges via buffer ov...,HIGH,7.8,NVD-CWE-Other,1999


In [10]:
print(f"Generating text embeddings with '{SBERT_MODEL_NAME}'...")
sbert_model = SentenceTransformer(SBERT_MODEL_NAME, device='cuda')

descriptions = df["Description"].tolist()
text_embeddings = sbert_model.encode(descriptions, show_progress_bar=True)

print(f"Text embeddings created with shape: {text_embeddings.shape}")

Generating text embeddings with 'all-MiniLM-L6-v2'...


c:\Python311\Lib\site-packages\huggingface_hub\file_download.py:945: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Batches: 100%|██████████| 226/226 [00:05<00:00, 38.75it/s]

Text embeddings created with shape: (7212, 384)


In [11]:
print("Generating categorical features for CWE...")
cwe_encoder = OneHotEncoder(handle_unknown='ignore')
cwe_features = cwe_encoder.fit_transform(df[["CWE"]])
print(f"CWE features created with shape: {cwe_features.shape}")

Generating categorical features for CWE...
CWE features created with shape: (7212, 317)


In [12]:
print("Generating numeric features for CVSS Score...")
scaler = StandardScaler()
numeric_features = scaler.fit_transform(df[["CVSS_Score"]])
print(f"Numeric features created with shape: {numeric_features.shape}")

Generating numeric features for CVSS Score...
Numeric features created with shape: (7212, 1)


In [13]:
print("Combining all features into a final matrix...")
X = np.hstack([
    text_embeddings,
    cwe_features.toarray(),
    numeric_features
])

y = df["Severity"]

print(f"Final feature matrix `X` created with shape: {X.shape}")
print(f"Target variable `y` created with shape: {y.shape}")

Combining all features into a final matrix...
Final feature matrix `X` created with shape: (7212, 702)
Target variable `y` created with shape: (7212,)


In [14]:
print("Saving features and preprocessing objects...")
joblib.dump((X, y), FEATURES_DATA_DIR / "features.pkl")
joblib.dump(cwe_encoder, FEATURES_DATA_DIR / "cwe_encoder.pkl")
joblib.dump(scaler, FEATURES_DATA_DIR / "scaler.pkl")

print("\nFeature engineering complete!")

Saving features and preprocessing objects...

Feature engineering complete!
